In [3]:
import pymupdf
import jinja2

In [4]:
# text blocks

with pymupdf.open("1p36 deletie syndroom Dutch FTNW.pdf") as doc:  # open document
    text = ''.join([block[4] for page in doc for block in page.get_text("blocks")])

print(text)

with open("output.txt", "w", encoding="utf8") as out:
    out.write(text)

 
 
 
 
rarechromo.org 
1p36 deletie 
syndroom  
2 
 
1p36 deletie syndroom 
Een 1p36 deletie is een zeldzame aandoening. Bij deze aandoening ontbreekt een stukje van 
chromosoom nummer 1. Voor een gezonde ontwikkeling van het ongeboren kind moet er de 
juiste hoeveelheid van de chromosomen aanwezig zijn, niet te veel en niet te weinig. Het 
ontbrekende stukje chromosoom 1 geeft meer kans op ontwikkelingsachterstand, 
leerproblemen, voedingsproblemen en aangeboren aandoeningen. 
 
Wat zijn chromosomen? 
Chromosomen bestaan uit erfelijk materiaal, het DNA. Het DNA bestaat uit genen waarin al 
onze erfelijke eigenschappen zijn vastgelegd. De genen ‘vertellen’ het lichaam hoe het zich 
moet ontwikkelen, groeien en functioneren.   
Chromosomen zitten in de lichaamscellen. De lichaamscellen zijn de bouwstenen van het 
lichaam. Elke lichaamscel heeft 23 paren chromosomen. Dat zijn in totaal 46 chromosomen. 
Van elk paar komt één chromosoom van de vader en één van de 
moeder.  
Van de 23 pare

In [5]:
# these examples are only from 1p36 deletie syndroom, doesn't overlap fully with all other pdfs

# body text
"""
    {'spans': [{'size': 9.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN',
       'color': 0,
       'alpha': 255,
"""

# header
"""
{'spans': [{'size': 14.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
"""

# quotes use these characters + wingdings font
# '\uf07d' (0x7D)
# '\uf07e' (0x7E)

# bullet points wingdings (filled square)
# '\uf06e ' (0x6E)
# 'size': 12.0

# For wingdings reference: https://www.alanwood.net/demos/wingdings.html

def is_body(span):
    return span["font"] == "DIN" and span["color"] == 0 and abs(span["size"] - 9) < 0.5
def is_maybe_body(span):
    return span["font"] == "DIN" and abs(span["size"] - 9) < 0.5
def is_header(span):
    return span["font"] == "DIN Medium" and span["color"] == 23466 and abs(span["size"] - 14) < 0.5
def is_quotation(span, text):
    return span["font"] == "Wingdings" and (text == '\uf07d' or text == '\uf07e')
def is_bullet(span, text):
    return span["font"] == "Wingdings" and abs(span["size"] - 12) < 0.1 and text == '\uf06e'

# bbox = tuple with top left point and bottom right point of rect
# bounding box containing the content of span
def grow_bbox(old_bbox, new_bbox):
    if old_bbox == None:
        return new_bbox
    top_left = (min(old_bbox[0], new_bbox[0]), min(old_bbox[1], new_bbox[1]))
    bottom_right = (max(old_bbox[2], new_bbox[2]), max(old_bbox[3], new_bbox[3]))
    return (top_left[0], top_left[1], bottom_right[0], bottom_right[1])

def bbox_overlaps(bbox1, bbox2):
    if bbox1[0] > bbox2[2] or bbox1[1] > bbox2[3]:
        return False
    if bbox2[0] > bbox1[2] or bbox2[1] > bbox1[3]:
        return False
    return True

def expand_bbox(bbox, extend):
    return (bbox[0] - extend, bbox[1] - extend, bbox[2] + extend, bbox[3] + extend)
    

sections = {}
with pymupdf.open("1p36 deletie syndroom Dutch FTNW.pdf") as doc:

    paragraphs = []
    paragraph = []
    last_paragraph_bbox = None
    bullets = []
    current_header = None

    for page in doc:
        quote_bboxes = []
        paragraph_bbox = None
        last_bullet_bbox = None
        first_text_bullet = False

        page_dict = page.get_text("dict", flags=18, sort=True)
        for block in page_dict["blocks"]:
            for line in block["lines"]:
                for span in line["spans"]:

                    bbox = span["bbox"]
                    text = span["text"].strip()
                    
                    if text.isspace() or text == "":
                        continue

                    if is_quotation(span, text):
                        quote_bboxes.append(bbox)
                        continue

                    if is_bullet(span, text):
                        last_bullet_bbox = bbox
                        first_text_bullet = True
                        continue

                    if is_body(span) and last_bullet_bbox is not None and bbox_overlaps(expand_bbox(bbox, 8), last_bullet_bbox):
                        if (first_text_bullet):
                            bullets.append(text)
                            first_text_bullet = False
                        else:
                            bullets[-1] = bullets[-1] + " " + text 
                        continue
                    elif len(bullets) > 0:
                        paragraphs.append(bullets)
                        bullets = []

                    if is_body(span) or is_maybe_body(span):
                        if (not is_body(span) and is_maybe_body(span) and paragraph_bbox is None):
                            if (any([bbox_overlaps(expand_bbox(bbox, 6), quote_bbox) for quote_bbox in quote_bboxes])):
                                continue
                            if (last_paragraph_bbox is not None and not bbox_overlaps(expand_bbox(bbox, 8), last_paragraph_bbox)):
                                continue

                        if (is_body(span) and paragraph_bbox is not None and not bbox_overlaps(expand_bbox(bbox, 8), paragraph_bbox)):
                            paragraphs.append(" ".join(paragraph))
                            paragraph = []
                            last_paragraph_bbox = paragraph_bbox
                            paragraph_bbox = None
                        
                        paragraph_bbox = grow_bbox(paragraph_bbox, bbox)
                        paragraph.append(text)                        
                        if text.endswith("."):
                            if ((bbox[2] - paragraph_bbox[0]) / (paragraph_bbox[2] - paragraph_bbox[0]) < 0.85):
                                paragraphs.append(" ".join(paragraph))
                                paragraph = []
                                last_paragraph_bbox = paragraph_bbox
                                paragraph_bbox = None

                    elif is_header(span):
                        if (current_header is not None):
                            if len(paragraph) > 0:
                                paragraphs.append(" ".join(paragraph))
                            sections[current_header] = paragraphs
                        paragraph = []
                        paragraphs = []
                        last_paragraph_bbox = paragraph_bbox
                        paragraph_bbox = None
                        current_header = text
                        continue

                    last_bullet_bbox = None

In [ ]:
# template 1: without dropdown accordion
templateLoader = jinja2.FileSystemLoader(searchpath="./")
templateEnv = jinja2.Environment(loader=templateLoader)
template = templateEnv.get_template("template.jinja")
outputHtml = template.render(sections = sections)

with open("output.html", "w", encoding="utf8") as out:
    out.write(outputHtml)

In [ ]:
# template 2: with dropdown accordion using html details element
templateLoader = jinja2.FileSystemLoader(searchpath="./")
templateEnv = jinja2.Environment(loader=templateLoader)
template = templateEnv.get_template("template2.jinja")
outputHtml = template.render(sections = sections)

with open("output_details.html", "w", encoding="utf8") as out:
    out.write(outputHtml)

In [ ]:
with pymupdf.open("1p36 deletie syndroom Dutch FTNW.pdf") as doc:
    for page in doc:
        display(page.get_text("dict", flags=18))

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (209.5, 572.0712280273438, 211.89999389648438, 582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': ' ',
       'origin': (209.5, 579.7999877929688),
       'bbox': (209.5,
        572.0712280273438,
        211.89999389648438,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (209.5,
      572.0712280273438,
      211.89999389648438,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flag

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (206.89999389648438,
    572.0712280273438,
    214.6999969482422,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '2 ',
       'origin': (206.89999389648438, 579.7999877929688),
       'bbox': (206.89999389648438,
        572.0712280273438,
        214.6999969482422,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (206.89999389648438,
      572.0712280273438,
      214.6999969482422,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size'

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (206.89999389648438,
    572.0712280273438,
    214.6999969482422,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '3 ',
       'origin': (206.89999389648438, 579.7999877929688),
       'bbox': (206.89999389648438,
        572.0712280273438,
        214.6999969482422,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (206.89999389648438,
      572.0712280273438,
      214.6999969482422,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size'

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (206.89999389648438,
    572.0712280273438,
    214.6999969482422,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '4 ',
       'origin': (206.89999389648438, 579.7999877929688),
       'bbox': (206.89999389648438,
        572.0712280273438,
        214.6999969482422,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (206.89999389648438,
      572.0712280273438,
      214.6999969482422,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size'

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (206.89999389648438,
    572.0712280273438,
    214.6999969482422,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '5 ',
       'origin': (206.89999389648438, 579.7999877929688),
       'bbox': (206.89999389648438,
        572.0712280273438,
        214.6999969482422,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (206.89999389648438,
      572.0712280273438,
      214.6999969482422,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size'

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (206.89999389648438,
    572.0712280273438,
    214.6999969482422,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '6 ',
       'origin': (206.89999389648438, 579.7999877929688),
       'bbox': (206.89999389648438,
        572.0712280273438,
        214.6999969482422,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (206.89999389648438,
      572.0712280273438,
      214.6999969482422,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'size'

... (8 outputs left out)

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (204.10000610351562,
    572.0712280273438,
    217.29998779296875,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '15 ',
       'origin': (204.10000610351562, 579.7999877929688),
       'bbox': (204.10000610351562,
        572.0712280273438,
        217.29998779296875,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (204.10000610351562,
      572.0712280273438,
      217.29998779296875,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'s

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (204.10000610351562,
    572.0712280273438,
    217.29998779296875,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '16 ',
       'origin': (204.10000610351562, 579.7999877929688),
       'bbox': (204.10000610351562,
        572.0712280273438,
        217.29998779296875,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (204.10000610351562,
      572.0712280273438,
      217.29998779296875,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'s

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (204.10000610351562,
    572.0712280273438,
    217.29998779296875,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '17 ',
       'origin': (204.10000610351562, 579.7999877929688),
       'bbox': (204.10000610351562,
        572.0712280273438,
        217.29998779296875,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (204.10000610351562,
      572.0712280273438,
      217.29998779296875,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'s

{'width': 419.6000061035156,
 'height': 595.2000122070312,
 'blocks': [{'type': 0,
   'number': 0,
   'flags': 0,
   'bbox': (204.10000610351562,
    572.0712280273438,
    217.29998779296875,
    582.0712280273438),
   'lines': [{'spans': [{'size': 10.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'DIN Medium',
       'color': 23466,
       'alpha': 255,
       'ascender': 0.7179999947547913,
       'descender': -0.210999995470047,
       'text': '18 ',
       'origin': (204.10000610351562, 579.7999877929688),
       'bbox': (204.10000610351562,
        572.0712280273438,
        217.29998779296875,
        582.0712280273438)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (204.10000610351562,
      572.0712280273438,
      217.29998779296875,
      582.0712280273438)}]},
  {'type': 0,
   'number': 1,
   'flags': 0,
   'bbox': (31.024999618530273,
    9.69003677368164,
    33.525001525878906,
    20.76003646850586),
   'lines': [{'spans': [{'s